# NBA Sports Betting Backtester

**CUIC Quant Fund — Semester 2 Project**

This notebook demonstrates the end-to-end backtesting framework for evaluating NBA sports betting strategies.  
It follows the architecture of [georgedouzas/sports-betting](https://github.com/georgedouzas/sports-betting) adapted for our NBA data pipeline.

---

## Contents

1. [Setup & Imports](#1-setup)
2. [Load NBA Data](#2-data)
3. [Single Backtest Run (Sanity Check)](#3-single)
4. [Walk-Forward Analysis with Model Refitting](#4-walkforward)
5. [Validation — 12-Check Suite](#5-validation)
6. [Performance Metrics](#6-metrics)
7. [Kelly Criterion Analysis](#7-kelly)
8. [Strategy Comparison](#8-comparison)
9. [Statistical Significance Tests](#9-stats)
10. [Visualisations](#10-viz)
11. [Edge Cases & Robustness Tests](#11-edgecases)
12. [Assumptions & Limitations](#12-assumptions)

---

> **Validation is key.** Every result in this notebook is validated against known
> baselines before being trusted.  Real bookmaker odds (Oct 2024 – present) are used
> where available; synthetic 50/50 odds are used for older games.

## 1. Setup & Imports <a id='1-setup'></a>

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

# Add project root to path if running from research/notebooks/
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

# Backtester imports
from cuic_quant.backtest import (
    load_nba_dataset,
    BacktestDataset,
    LogisticRegressionStrategy,
    RandomForestStrategy,
    HomeAdvantageBaseline,
    ValueBetBaseline,
    TransactionCosts,
    BacktestConfig,
    run_backtest,
    run_walk_forward,
    SimpleSplit,
    RollingWindow,
    ExpandingWindow,
    CPCV,
    BacktestValidator,
    compare_strategies,
    detect_anomalies,
    rank_strategies,
    binomial_test,
    bootstrap_ci,
    bonferroni_correction,
    holm_bonferroni_correction,
    deflated_sharpe_ratio,
    probability_of_backtest_overfitting,
    equity_curve,
    drawdown_plot,
    rolling_sharpe,
    bet_distribution,
    edge_scatter,
    metrics_bar_chart,
    fold_performance,
)
from cuic_quant.metrics import calculate_all_metrics

print('All imports successful ✓')

## 2. Load NBA Data <a id='2-data'></a>

The data loader attempts to load real NBA CSVs from `data/nba_collection/`.  
If they are not available, it automatically falls back to a deterministic synthetic dataset  
with a built-in home-court edge (~55% home win rate) so the backtester can run end-to-end.

In [ ]:
# Load dataset — real NBA games with real bookmaker odds where available
# real_odds_only=True restricts to the 932 games that have real multi-bookmaker odds
dataset = load_nba_dataset(
    data_dir='../../data/nba_collection',
    processed_dir='../../data/processed',
    odds_path='../../data/odds/game_odds.csv',
    real_odds_only=True,   # Use only games with real bookmaker odds
)

X, Y, O, dates = dataset.X, dataset.Y, dataset.O, dataset.dates

print(f'Dataset loaded:')
print(f'  Games (rows):     {len(X):,}')
print(f'  Features (cols):  {X.shape[1]}')
print(f'  Home win rate:    {Y.mean():.1%}')
print(f'  Date range:       {dates.min().date()} → {dates.max().date()}')
print(f'  Avg home odds:    {O["home_odds"].mean():.3f}')
print(f'  Avg away odds:    {O["away_odds"].mean():.3f}')

# Implied vig check
avg_vig = (1/O['home_odds'] + 1/O['away_odds']).mean() - 1
print(f'  Avg bookmaker vig: {avg_vig:.1%}')
print()
print('✓ Using real multi-bookmaker odds (DraftKings, FanDuel, Caesars, etc.)')

In [ ]:
# Preview feature matrix
print('Feature sample:')
X.head(3)

In [ ]:
# Preview odds
print('Odds sample:')
O.head(3)

## 3. Single Backtest Run (Sanity Check) <a id='3-single'></a>

Before running walk-forward, we run a **simple train/test split** to verify:
- The engine produces trades with the correct schema
- A strategy with edge (LogReg trained on synthetic features) shows profit  
- A zero-edge baseline (HomeAdvantage) approximately breaks even

This is our **known-edge validation** — if this fails, the engine has a bug.

In [ ]:
# 80/20 train-test split
split_idx = int(len(X) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
Y_train, Y_test = Y.iloc[:split_idx], Y.iloc[split_idx:]
O_test = O.iloc[split_idx:]
dates_test = dates[split_idx:]

config = BacktestConfig(
    initial_bankroll=10_000.0,
    kelly_fraction=0.5,          # Half Kelly — conservative
    max_bet_fraction=0.10,       # Max 10% of bankroll per bet
    min_edge=0.02,               # Require at least 2% edge to bet
    transaction_costs=TransactionCosts(commission_pct=0.01),  # 1% commission
)

# Strategy 1: Logistic Regression
logreg = LogisticRegressionStrategy(C=0.5)
trades_logreg = run_backtest(
    strategy=logreg,
    X_train=X_train, Y_train=Y_train,
    X_test=X_test, Y_test=Y_test,
    O_test=O_test,
    config=config,
    dates_test=dates_test,
    fold_id=0,
)

# Strategy 2: Home Advantage Baseline
baseline = HomeAdvantageBaseline()
trades_baseline = run_backtest(
    strategy=baseline,
    X_train=X_train, Y_train=Y_train,
    X_test=X_test, Y_test=Y_test,
    O_test=O_test,
    config=config,
    dates_test=dates_test,
    fold_id=0,
)

print(f'LogReg:   {len(trades_logreg):3d} bets, total P&L = £{trades_logreg["pnl"].sum():.2f}')
print(f'Baseline: {len(trades_baseline):3d} bets, total P&L = £{trades_baseline["pnl"].sum():.2f}')

trades_logreg.head()

## 4. Walk-Forward Analysis with Model Refitting <a id='4-walkforward'></a>

Walk-forward backtesting prevents lookahead bias by **refitting the model on each fold**.  
We use an **expanding window**: the training set grows with each fold, the model is retrained,  
and predictions are made only on out-of-sample data.

**Refitting is critical** — using a single model trained on all data would implicitly  
use future information to generate historical predictions.

In [ ]:
# Expanding window walk-forward (the recommended default)
splitter = ExpandingWindow(
    min_train_size=150,   # Need at least 150 games to train
    test_size=50,         # Evaluate on 50 games per fold
    step=50,              # Advance 50 games each fold
)

# Show how the splits look
print('Walk-forward fold summary:')
print(f'{"Fold":>5}  {"Train":>8}  {"Test":>8}')
for i, (tr, te) in enumerate(splitter.split(len(X))):
    print(f'{i:>5}  {len(tr):>8}  {len(te):>8}')
print()

In [ ]:
# Run walk-forward for all strategies
strategies = {
    'LogisticRegression': LogisticRegressionStrategy(C=0.5),
    'RandomForest':       RandomForestStrategy(n_estimators=100, max_depth=4),
    'HomeBaseline':       HomeAdvantageBaseline(),
    'ValueBetBaseline':   ValueBetBaseline(implied_prob_threshold=0.52),
}

wf_config = BacktestConfig(
    initial_bankroll=10_000.0,
    kelly_fraction=0.5,
    max_bet_fraction=0.10,
    min_edge=0.02,
    transaction_costs=TransactionCosts(commission_pct=0.01),
)

wf_results = {}
for name, strat in strategies.items():
    print(f'Running walk-forward: {name}...')
    trades = run_walk_forward(strat, dataset, ExpandingWindow(), wf_config)
    wf_results[name] = trades
    n_bets = len(trades)
    total_pnl = trades['pnl'].sum() if n_bets > 0 else 0.0
    print(f'  {n_bets:4d} bets | Total P&L: £{total_pnl:,.2f}')

print('\n✓ Walk-forward complete')

### Comparison: Split strategies

We also show results using **Rolling Window** (fixed-size) and **CPCV** for comparison.

In [ ]:
# Compare split strategies on LogisticRegression
lr_strategy = LogisticRegressionStrategy(C=0.5)

split_results = {}
for split_name, splitter_obj in [
    ('SimpleSplit',      SimpleSplit(test_fraction=0.2)),
    ('RollingWindow',    RollingWindow(train_size=200, test_size=50, step=50)),
    ('ExpandingWindow',  ExpandingWindow(min_train_size=150, test_size=50, step=50)),
    ('CPCV',             CPCV(n_splits=6, n_test_splits=2)),
]:
    from cuic_quant.backtest.engine import LogisticRegressionStrategy as LRS  # fresh copy each time
    trades = run_walk_forward(LRS(C=0.5), dataset, splitter_obj, wf_config)
    split_results[split_name] = trades
    print(f'{split_name:20s}: {len(trades):4d} bets | P&L £{trades["pnl"].sum():.2f}')

## 5. Validation — 12-Check Suite <a id='5-validation'></a>

Before trusting any numbers, we run the 12-check validator on the best strategy.  
**All error-severity checks must pass** before metrics are used.

In [ ]:
validator = BacktestValidator(
    max_bet_fraction=wf_config.max_bet_fraction,
)

# Validate best performing strategy
best_strategy_name = 'LogisticRegression'
best_trades = wf_results[best_strategy_name]

validation_summary = validator.summary(best_trades)

# Style the table
def colour_passed(val):
    if val == '✓': return 'background-color: #d4edda; color: #155724'
    if val == '✗': return 'background-color: #f8d7da; color: #721c24'
    return ''

print(f'Validation results for {best_strategy_name}:')
print(f'All error checks passed: {validator.all_passed(best_trades)}')
print()
validation_summary.style.map(colour_passed, subset=['passed'])

In [ ]:
# Validate all strategies
print('Validation pass/fail summary:')
print(f'{"Strategy":20s}  {"All Passed":10s}  {"Bets":6s}')
print('-' * 45)
for name, trades in wf_results.items():
    all_ok = validator.all_passed(trades) if len(trades) > 0 else False
    status = '✓' if all_ok else '✗'
    print(f'{name:20s}  {status:10s}  {len(trades):6d}')

## 6. Performance Metrics <a id='6-metrics'></a>

Full metrics suite including: Sharpe, Sortino, Calmar, Max Drawdown, Win Rate,  
Profit Factor, ROI, Brier Score, Log Loss, and Kelly Growth Rate.

In [ ]:
print('=== Performance Metrics — LogisticRegression Walk-Forward ===')
print()

metrics = calculate_all_metrics(best_trades)

# Format nicely
display_names = {
    'total_trades':     'Total Trades',
    'win_rate':         'Win Rate',
    'total_pnl':        'Total P&L (£)',
    'roi':              'ROI',
    'sharpe_ratio':     'Sharpe Ratio (ann.)',
    'sortino_ratio':    'Sortino Ratio (ann.)',
    'calmar_ratio':     'Calmar Ratio',
    'max_drawdown':     'Max Drawdown',
    'profit_factor':    'Profit Factor',
    'brier_score':      'Brier Score (lower=better)',
    'log_loss':         'Log Loss (lower=better)',
    'kelly_growth_rate':'Kelly Growth Rate (per bet)',
}

pct_metrics = {'win_rate', 'roi', 'max_drawdown'}

rows = []
for key, label in display_names.items():
    if key not in metrics:
        continue
    val = metrics[key]
    if key in pct_metrics:
        fmt = f'{val:.1%}'
    elif key == 'total_trades':
        fmt = str(int(val))
    elif key == 'total_pnl':
        fmt = f'£{val:,.2f}'
    else:
        fmt = f'{val:.4f}'
    rows.append({'Metric': label, 'Value': fmt})

pd.DataFrame(rows).set_index('Metric')

## 7. Kelly Criterion Analysis <a id='7-kelly'></a>

The **Kelly Criterion** determines the optimal fraction of your bankroll to bet on each game  
to maximise long-run bankroll growth. Betting the full Kelly maximises geometric growth but  
leads to very large drawdowns; we use **Half Kelly** (50%) for a better risk/reward balance.

### Formula

$$f^* = \frac{p \cdot b - q}{b}$$

Where:
- $f^*$ = fraction of bankroll to bet  
- $p$ = estimated probability of winning (from our model)  
- $q = 1 - p$ = probability of losing  
- $b$ = decimal odds $- 1$ (net profit per unit staked)

### Key Properties
- **Full Kelly** → maximises expected log(wealth), but ~30% max drawdown expected
- **Half Kelly** → ~75% of Kelly growth rate, ~15% expected max drawdown  
- **Kelly = 0** → no edge; negative Kelly = do not bet
- **Over-betting Kelly** → negative expected log growth (ruin possible)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 7a. Kelly formula visualisation ---
def kelly_fraction(p: float, decimal_odds: float) -> float:
    """Full Kelly fraction given win probability and decimal odds."""
    b = decimal_odds - 1          # net profit per unit
    q = 1 - p
    return max(0.0, (p * b - q) / b)

# Show Kelly fractions across a range of probabilities and odds
probs = np.linspace(0.35, 0.75, 200)
odds_scenarios = [1.5, 1.91, 2.5, 3.5]

fig, ax = plt.subplots(figsize=(9, 5))
for odds in odds_scenarios:
    fracs = [kelly_fraction(p, odds) for p in probs]
    implied = 1 / odds
    ax.plot(probs * 100, [f * 100 for f in fracs], label=f'Odds {odds:.2f} (implied {implied:.1%})')
    ax.axvline(implied * 100, color='grey', linestyle=':', alpha=0.4)

ax.axhline(10, color='red', linestyle='--', linewidth=0.8, label='Max bet cap (10%)')
ax.set_xlabel('Model win probability (%)')
ax.set_ylabel('Full Kelly stake (% of bankroll)')
ax.set_title('Kelly Criterion — Optimal Stake vs Win Probability')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Kelly fractions at typical NBA edges:')
print(f'{"Prob":>6}  {"Odds":>6}  {"Edge":>8}  {"Full Kelly":>12}  {"Half Kelly":>12}')
print('-' * 55)
for p, o in [(0.55, 1.91), (0.60, 2.10), (0.65, 2.50), (0.70, 3.00)]:
    implied = 1/o
    edge = p - implied
    fk = kelly_fraction(p, o)
    print(f'{p:>6.0%}  {o:>6.2f}  {edge:>+8.1%}  {fk:>12.1%}  {fk/2:>12.1%}')

In [ ]:
# --- 7b. Kelly stake distribution from real backtest trades ---
if 'stake' in best_trades.columns and 'bankroll' in best_trades.columns:
    kelly_stakes = best_trades['stake'] / best_trades['bankroll'].shift(1).fillna(wf_config.initial_bankroll)
    
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    
    # Stake as % of bankroll
    axes[0].hist(kelly_stakes * 100, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    axes[0].axvline(kelly_stakes.mean() * 100, color='red', linestyle='--', label=f'Mean {kelly_stakes.mean():.1%}')
    axes[0].axvline(wf_config.max_bet_fraction * 100, color='orange', linestyle='--', label=f'Cap {wf_config.max_bet_fraction:.0%}')
    axes[0].set_xlabel('Stake as % of bankroll')
    axes[0].set_ylabel('Number of bets')
    axes[0].set_title('Kelly Stake Distribution (Half Kelly × cap)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Kelly stake vs edge scatter
    axes[1].scatter(best_trades['edge'] * 100, kelly_stakes * 100, alpha=0.4, s=15, color='steelblue')
    axes[1].set_xlabel('Model edge (%)')
    axes[1].set_ylabel('Stake as % of bankroll')
    axes[1].set_title('Edge vs Kelly Stake Size')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f'Kelly stake statistics (as % of bankroll):')
    print(f'  Mean stake:   {kelly_stakes.mean():.2%}')
    print(f'  Max stake:    {kelly_stakes.max():.2%}')
    print(f'  Bets at cap:  {(kelly_stakes >= wf_config.max_bet_fraction * 0.99).mean():.1%} of all bets')
else:
    print('Stake/bankroll columns not found in trades.')

## 8. Strategy Comparison <a id='8-comparison'></a>

Side-by-side comparison of all strategies with composite ranking.

In [ ]:
# Full comparison table
comparison = compare_strategies(wf_results, primary_metric='sharpe_ratio')
print('Strategy Comparison (sorted by Sharpe):')
comparison[['total_trades', 'win_rate', 'total_pnl', 'sharpe_ratio',
            'max_drawdown', 'profit_factor', 'roi']].round(4)

In [ ]:
# Composite ranking
ranked = rank_strategies(wf_results)
print('Composite Ranking:')
ranked[['sharpe_ratio', 'win_rate', 'roi', 'max_drawdown', 'composite_score', 'rank']].round(4)

In [ ]:
# Anomaly detection — flag suspiciously extreme results
print('Anomaly Detection:')
anomalies = detect_anomalies(wf_results)
if anomalies.empty:
    print('  No anomalies detected ✓')
else:
    print(f'  {len(anomalies)} anomalies flagged — investigate before trusting results!')
    anomalies

## 9. Statistical Significance Tests <a id='9-stats'></a>

We apply rigorous statistical tests to assess whether the observed performance  
is genuine alpha or could be explained by luck:

- **Binomial test**: Is the win rate significantly above the implied probability?
- **Bootstrap CI**: 95% confidence interval on mean P&L per bet
- **Bonferroni / Holm-Bonferroni**: Correct for testing multiple strategies
- **Deflated Sharpe Ratio (DSR)**: Adjust for multiple strategy trials
- **PBO**: Probability of Backtest Overfitting

In [ ]:
# Binomial test: is win rate > implied probability?
print('=== Binomial Test (LogisticRegression) ===')
btest = binomial_test(best_trades, alpha=0.05)
for k, v in btest.items():
    print(f'  {k:20s}: {v}')

In [ ]:
# Bootstrap 95% CI on mean P&L per bet
print('=== Bootstrap 95% CI on Mean P&L Per Bet ===')
ci = bootstrap_ci(
    best_trades['pnl'],
    stat_fn=lambda s: float(s.mean()),
    n_bootstrap=2000,
    ci_level=0.95,
)
print(f'  Point estimate: £{ci["point_estimate"]:.4f}')
print(f'  95% CI:         £{ci["lower"]:.4f} — £{ci["upper"]:.4f}')
print(f'  Positive at 95% confidence: {ci["lower"] > 0}')

In [ ]:
# Multiple testing correction across strategies
print('=== Multiple Testing Correction (Holm-Bonferroni) ===')
p_values = []
strategy_names = []
for name, trades in wf_results.items():
    if len(trades) > 0:
        bt = binomial_test(trades)
        p_values.append(bt['p_value'])
        strategy_names.append(name)

holm = holm_bonferroni_correction(p_values)
holm.insert(0, 'strategy', strategy_names)
print(holm.to_string(index=False))

In [ ]:
# Deflated Sharpe Ratio
print('=== Deflated Sharpe Ratio ===')
n_strategies = len(strategies)  # Number of strategies tried
sr = metrics.get('sharpe_ratio', 0.0)
n_obs = len(best_trades)

pnl_series = pd.to_numeric(best_trades['pnl'], errors='coerce').dropna()
skew = float(pnl_series.skew()) if len(pnl_series) > 2 else 0.0
kurt = float(pnl_series.kurtosis() + 3) if len(pnl_series) > 3 else 3.0

dsr = deflated_sharpe_ratio(sr, n_trials=n_strategies, n_observations=n_obs, skewness=skew, kurtosis=kurt)
print(f'  Observed Sharpe:  {sr:.4f}')
print(f'  Strategies tried: {n_strategies}')
print(f'  Return skewness:  {skew:.4f}')
print(f'  DSR (prob true Sharpe > 0): {dsr:.4f}')
print(f'  Interpretation: {"Strong signal" if dsr > 0.95 else "Moderate" if dsr > 0.8 else "Weak — could be luck"}')

In [ ]:
# PBO from fold Sharpes
print('=== Probability of Backtest Overfitting (PBO) ===')
from cuic_quant.metrics import calculate_sharpe_ratio

fold_sharpes = []
if 'fold' in best_trades.columns:
    for fold_id in sorted(best_trades['fold'].unique()):
        fold_trades = best_trades[best_trades['fold'] == fold_id]
        fold_sharpes.append(calculate_sharpe_ratio(fold_trades['pnl']))
    
    pbo = probability_of_backtest_overfitting(fold_sharpes)
    print(f'  Per-fold Sharpes: {[round(s, 3) for s in fold_sharpes]}')
    print(f'  PBO estimate: {pbo:.2f}')
    print(f'  Interpretation: {"Likely overfitting" if pbo > 0.5 else "No overfitting signal"}')
else:
    print('  Fold column not found in trades DataFrame')

## 10. Visualisations <a id='10-viz'></a>

In [ ]:
# --- Equity curves: all strategies ---
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Equity Curves — Walk-Forward Results', fontsize=13)

colors = ['steelblue', 'darkorange', 'seagreen', 'crimson']
for ax, (name, trades), color in zip(axes.flatten(), wf_results.items(), colors):
    if trades.empty:
        ax.text(0.5, 0.5, f'{name}\nNo trades', ha='center', va='center')
        continue
    initial = wf_config.initial_bankroll
    if 'bankroll' in trades.columns:
        equity = pd.concat([pd.Series([initial]), trades['bankroll'].reset_index(drop=True)])
    else:
        equity = initial + pd.concat([pd.Series([0.0]), trades['cumulative_pnl'].reset_index(drop=True)])
    
    final = float(equity.iloc[-1])
    pct = (final - initial) / initial * 100
    ax.plot(equity.values, color=color, linewidth=1.5)
    ax.axhline(initial, color='grey', linestyle='--', alpha=0.5, linewidth=0.8)
    ax.set_title(f'{name}  ({pct:+.1f}%)', fontsize=10)
    ax.set_xlabel('Trade #', fontsize=8)
    ax.set_ylabel('Bankroll (£)', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- Drawdown plot ---
fig = drawdown_plot(best_trades, initial_bankroll=wf_config.initial_bankroll,
                    title=f'Drawdown — {best_strategy_name}')
plt.show()

In [ ]:
# --- Rolling Sharpe ---
fig = rolling_sharpe(best_trades, window=30,
                     title=f'Rolling Sharpe (window=30) — {best_strategy_name}')
plt.show()

In [ ]:
# --- Bet distribution ---
fig = bet_distribution(best_trades, title=f'Bet Distribution — {best_strategy_name}')
plt.show()

In [ ]:
# --- Edge vs P&L scatter ---
fig = edge_scatter(best_trades, title=f'Edge vs P&L — {best_strategy_name}')
plt.show()

In [ ]:
# --- Metrics bar chart ---
comparison = compare_strategies(wf_results)
fig = metrics_bar_chart(comparison, metrics=['sharpe_ratio', 'win_rate', 'roi', 'profit_factor'],
                        title='Strategy Comparison')
plt.show()

In [ ]:
# --- Fold performance ---
fig = fold_performance(best_trades, title=f'Per-Fold P&L — {best_strategy_name}')
plt.show()

## 11. Edge Cases & Robustness Tests <a id='11-edgecases'></a>

The backtester must handle these edge cases without crashing or producing nonsense:

In [ ]:
from cuic_quant.backtest.data_loader import _generate_synthetic
from cuic_quant.backtest.engine import _empty_trades_df

edge_cases = {}

# === 1. Empty dataset ===
print('Test 1: Empty training set')
empty_ds = _generate_synthetic(n_games=10, seed=0)
t = run_backtest(
    LogisticRegressionStrategy(), X_train=empty_ds.X.iloc[:0], Y_train=empty_ds.Y.iloc[:0],
    X_test=empty_ds.X, Y_test=empty_ds.Y, O_test=empty_ds.O, fold_id=0,
)
edge_cases['empty_training'] = f'Handled: {len(t)} trades (expected 0)'
print(f'  {edge_cases["empty_training"]} ✓')

# === 2. Single game ===
print('Test 2: Single game in test set')
single_ds = _generate_synthetic(n_games=200, seed=1)
t = run_backtest(
    LogisticRegressionStrategy(), X_train=single_ds.X.iloc[:150], Y_train=single_ds.Y.iloc[:150],
    X_test=single_ds.X.iloc[150:151], Y_test=single_ds.Y.iloc[150:151],
    O_test=single_ds.O.iloc[150:151], fold_id=0,
)
edge_cases['single_game'] = f'Handled: {len(t)} trades'
print(f'  {edge_cases["single_game"]} ✓')

# === 3. Extreme odds (very high) ===
print('Test 3: Extreme odds (10.0 / 1.05)')
extreme_ds = _generate_synthetic(n_games=200, seed=2)
extreme_O = extreme_ds.O.copy()
extreme_O['home_odds'] = 10.0
extreme_O['away_odds'] = 1.05
extreme_ds_mod = BacktestDataset(extreme_ds.X, extreme_ds.Y, extreme_O, extreme_ds.dates)
t = run_walk_forward(LogisticRegressionStrategy(), extreme_ds_mod, ExpandingWindow(), wf_config)
edge_cases['extreme_odds'] = f'Handled: {len(t)} trades'
print(f'  {edge_cases["extreme_odds"]} ✓')

# === 4. Zero bankroll ===
print('Test 4: Near-zero bankroll config')
zero_config = BacktestConfig(initial_bankroll=0.01, kelly_fraction=0.5, min_edge=0.02)
normal_ds = _generate_synthetic(n_games=200, seed=3)
t = run_walk_forward(LogisticRegressionStrategy(), normal_ds, ExpandingWindow(), zero_config)
edge_cases['zero_bankroll'] = f'Handled: {len(t)} trades'
print(f'  {edge_cases["zero_bankroll"]} ✓')

# === 5. No edge — min_edge very high ===
print('Test 5: min_edge=0.99 (no bets expected)')
no_edge_config = BacktestConfig(initial_bankroll=10000, kelly_fraction=0.5, min_edge=0.99)
t = run_walk_forward(LogisticRegressionStrategy(), dataset, ExpandingWindow(), no_edge_config)
edge_cases['no_edge'] = f'Handled: {len(t)} trades (expected 0)'
print(f'  {edge_cases["no_edge"]} ✓')

# === 6. All same outcome (all home wins) ===
print('Test 6: All home wins in training data')
all_home_ds = _generate_synthetic(n_games=200, seed=4)
all_home_ds_mod = BacktestDataset(
    all_home_ds.X,
    all_home_ds.Y * 0 + 1,  # all home wins in train
    all_home_ds.O,
    all_home_ds.dates,
)
t = run_walk_forward(HomeAdvantageBaseline(), all_home_ds_mod, ExpandingWindow(), wf_config)
edge_cases['all_same_outcome'] = f'Handled: {len(t)} trades'
print(f'  {edge_cases["all_same_outcome"]} ✓')

print('\n✓ All edge cases handled without errors')

In [ ]:
# Validate edge case results (they may have 0 trades — that's fine)
print('Edge case validation summary:')
for case_name, result_str in edge_cases.items():
    print(f'  {case_name:25s}: {result_str}')

## 12. Assumptions & Limitations <a id='12-assumptions'></a>

### Assumptions

| Assumption | Justification |
|---|---|
| Decimal odds > 1.0 are valid bets | Invalid odds are skipped in the engine |
| Kelly criterion uses model P(home_win) | Assumes model is well-calibrated (check Brier score) |
| Transaction costs are proportional to stake | Simplification — actual costs may vary by bookmaker |
| No liquidity constraints | Real betting markets have bet limits |
| Odds at time of bet = opening odds | CLV metric requires closing odds for full validation |
| Features have no lookahead bias | Guaranteed by `PreGameFeatureBuilder` in `data/nba/features.py` |
| Model is refit on each fold | Walk-forward design prevents future information leakage |

### Limitations

1. **Calibration not guaranteed**: Logistic regression probabilities may not be perfectly calibrated. Use Brier score and log loss to assess.

2. **Vig not accounted in odds conversion**: We normalise implied probs by dividing by total implied probability — this approximates fair odds but ignores bookmaker margin variation.

3. **Closing Line Value (CLV)**: CLV requires closing odds data which is not in the current NBA dataset. This is the gold standard for edge validation and should be added when data is available.

4. **Correlation between bets**: Multiple bets on the same day are treated as independent. In reality, NBA game outcomes are correlated (same slate, fatigue, back-to-backs).

5. **Synthetic data**: Results on synthetic data show the *engine* works correctly, not that the strategy has real edge. Real validation requires real odds data.

6. **No line shopping**: The engine uses a single odds source. In practice, bet at the best available odds.

### Next Steps

- [ ] Integrate real closing odds data for CLV calculation
- [ ] Add gradient boosting (XGBoost/LightGBM) strategy
- [ ] Implement proper Platt scaling / isotonic regression for probability calibration
- [ ] Add correlated bet adjustment in Kelly sizing
- [ ] Back-test on multiple seasons to assess out-of-sample stability

In [ ]:
# Final summary
print('=' * 60)
print('CUIC NBA BACKTESTER — FINAL SUMMARY')
print('=' * 60)
print()
print(f'Dataset:      {len(X):,} games  ({dates.min().date()} → {dates.max().date()})')
print(f'Odds source:  Real bookmaker odds (DraftKings, FanDuel, Caesars, BetMGM, +more)')
print(f'Avg vig:      {((1/O["home_odds"] + 1/O["away_odds"]).mean() - 1):.1%}')
print(f'Strategies:   {len(strategies)}')
print(f'Splitter:     ExpandingWindow (carry bankroll across folds)')
print()

from cuic_quant.metrics import calculate_sharpe_ratio
print('Walk-forward results:')
for name, trades in wf_results.items():
    if trades.empty:
        print(f'  {name:20s}: no trades')
        continue
    pnl_total = trades['pnl'].sum()
    wr = (trades['outcome'].str.upper() == 'WIN').mean()
    sr = calculate_sharpe_ratio(trades['pnl'])
    print(f'  {name:20s}: {len(trades):4d} bets | P&L £{pnl_total:+8,.2f} | WR {wr:.1%} | SR {sr:.3f}')

print()
print(f'Validation:   All error checks passed ✓' if validator.all_passed(best_trades) else 'Validation:   FAILED ✗')
print()
print('Framework ready for live deployment ✓')
